# ChainValley — offline analysis

Read `artifacts/runs/*.json`, compute survival rounds, Gini, and violation attempt rate directly from run artifacts, then compare soft vs hard with Mann–Whitney U.

In [ ]:
from pathlib import Path
import json
from statistics import mean

import numpy as np
from scipy.stats import mannwhitneyu

runs_dir = Path('..') / 'artifacts' / 'runs'
records = []
if runs_dir.exists():
    for path in sorted(runs_dir.glob('*.json')):
        if path.name.endswith('.partial.json'):
            continue
        with open(path, encoding='utf-8') as f:
            rec = json.load(f)
            rec['_path'] = str(path)
            records.append(rec)
print(f'Loaded {len(records)} records from {runs_dir}')

In [ ]:
AGENT_CODES = ('A', 'B', 'C', 'D', 'E')

def gini(values):
    xs = np.asarray(values, dtype=float)
    if xs.size == 0 or np.all(xs == 0):
        return 0.0
    xs = np.sort(xs)
    n = len(xs)
    idx = np.arange(1, n + 1)
    return float((2 * np.sum(idx * xs)) / (n * np.sum(xs)) - (n + 1) / n)

def cumulative_from_harvests(rec):
    totals = {code: 0 for code in AGENT_CODES}
    for item in rec.get('harvests', []):
        totals[item['agent']] += int(item.get('executed', 0))
    return totals

def survival_rounds(rec):
    rounds = rec.get('rounds', [])
    if rounds:
        return len(rounds)
    return int((rec.get('metrics') or {}).get('survival_rounds', 0))

def violation_rate(rec):
    harvests = rec.get('harvests', [])
    if not harvests:
        return 0.0
    violations = sum(1 for item in harvests if int(item.get('requested', 0)) > 4)
    return violations / len(harvests)

def summarize_record(rec):
    totals = cumulative_from_harvests(rec)
    return {
        'path': rec.get('_path'),
        'condition': rec.get('condition'),
        'survival_rounds': survival_rounds(rec),
        'violation_rate': violation_rate(rec),
        'gini': gini([totals[c] for c in AGENT_CODES]),
        'totals': totals,
    }

summary = [summarize_record(rec) for rec in records]
summary[:2]

In [ ]:
soft = [row for row in summary if row['condition'] == 'soft']
hard = [row for row in summary if row['condition'] == 'hard']

EXPECTED_N = 10

def compare(metric):
    s = [row[metric] for row in soft]
    h = [row[metric] for row in hard]
    if len(s) != EXPECTED_N or len(h) != EXPECTED_N:
        return {
            'metric': metric,
            'ready': False,
            'message': f'Require exactly {EXPECTED_N} soft and {EXPECTED_N} hard runs before formal statistics',
            'soft_n': len(s),
            'hard_n': len(h),
            'u': None,
            'p': None,
        }
    u, p = mannwhitneyu(s, h, alternative='two-sided')
    return {
        'metric': metric,
        'ready': True,
        'soft_mean': mean(s),
        'hard_mean': mean(h),
        'soft_n': len(s),
        'hard_n': len(h),
        'u': float(u),
        'p': float(p),
    }

results = [compare('survival_rounds'), compare('violation_rate'), compare('gini')]
results